In [1]:
import yfinance as yf
import pandas as pd

In [2]:
dat = yf.Ticker("HNST")   

# Daily prices (adjusted), then take last trading day of each month
d = dat.history(period="17y", interval="1d", auto_adjust=True)

s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end
s.index.name = "month_end"

# Make month_end a normal column and add an integer index column
monthly = s.reset_index(name="adj_close")
monthly.insert(0, "idx", range(1, len(monthly)+1))   # 1-based; use range(len(monthly)) for 0-based

#Change column names
monthly=monthly.rename(columns={"month_end":"Date", "adj_close":"Price"})


#Drop idx column
monthly=monthly.drop(columns=["idx"])

#Convert to DataFrame
stock_price = pd.DataFrame(monthly)

# Add column called calendar_quarter

p = stock_price["Date"].dt.to_period("Q")        # calendar quarters
stock_price["cal_year"] = p.dt.year.astype("Int64")    # 2010, 2011, ...
stock_price["cal_q"]    = p.dt.quarter                 # 1..4

# Final label: "Q{cal_q} {cal_year}"
stock_price["calendar_quarter"] = "Q" + stock_price["cal_q"].astype(str) + " " + stock_price["cal_year"].astype(str)


#Drop cal_year and cal_q column
stock_price=stock_price.drop(columns=["cal_year", "cal_q"])


stock_price


/var/folders/_2/ksgd3nl52j9g0sxwpjjd1bgh0000gn/T/ipykernel_9499/2859816483.py:6: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  s = d["Close"].tz_localize(None).resample("M").last()   # Series indexed by month-end


,Date,Price,calendar_quarter
0,2021-05-31,15.780000,Q2 2021
1,2021-06-30,16.190001,Q2 2021
2,2021-07-31,14.370000,Q3 2021
3,2021-08-31,10.180000,Q3 2021
4,2021-09-30,10.380000,Q3 2021
5,2021-10-31,9.150000,Q4 2021
6,2021-11-30,8.540000,Q4 2021
7,2021-12-31,8.090000,Q4 2021
8,2022-01-31,6.490000,Q1 2022
9,2022-02-28,5.780000,Q1 2022


In [3]:
stock_price.to_csv("hnst_price.csv")